1: Connecting to libraries and loading data

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def load_dataset():
    data = load_breast_cancer(as_frame=True)
    return data

raw_data = load_dataset()
df = raw_data.frame

2:Dimensions, Feature Names, Class Labels, Missing Values, Distribution

In [2]:
# 1. Dataset dimensions
print("1. Dataset Dimensions (Rows, Columns):", df.shape)

# 2. Feature names
print("\n2. Feature Names (First 5):", list(raw_data.feature_names[:5]))
print("Total Features:", len(raw_data.feature_names))

# 3. Class labels
print("\n3. Class Labels:", list(raw_data.target_names)) # 0 = malignant, 1 = benign

# 4. Missing values
print("\n4. Total Missing Values:", df.isnull().sum().sum())

# 5. Class distribution
print("\n5. Class Distribution:")
print(df['target'].value_counts().rename(index={0: 'Malignant (0)', 1: 'Benign (1)'}))

df.head()

1. Dataset Dimensions (Rows, Columns): (569, 31)

2. Feature Names (First 5): ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness']
Total Features: 30

3. Class Labels: ['malignant', 'benign']

4. Total Missing Values: 0

5. Class Distribution:
target
Benign (1)       357
Malignant (0)    212
Name: count, dtype: int64


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


3.Splitting & Preprocessing + Checking Shapes

In [3]:
def split_and_preprocess(df, test_size=0.2, random_state=42):
    X = df.drop(columns=['target'])
    y = df['target']
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler, X.columns

X_train, X_test, y_train, y_test, scaler, feature_cols = split_and_preprocess(df)

# Check that feature shapes make sense
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (455, 30)
X_test shape: (114, 30)
y_train shape: (455,)
y_test shape: (114,)


4.Model Fitting

In [4]:
def train_model(X_train, y_train, random_state=42):
    model = LogisticRegression(random_state=random_state, max_iter=1000)
    model.fit(X_train, y_train)
    return model

model = train_model(X_train, y_train)
print("Model fitted successfully!")

Model fitted successfully!


5:Verify Results

In [8]:
def evaluate_model(model, X_test, y_test):
    """"Calculating model accuracy and displaying the evaluation report"""
   
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    
    print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)\n")
    print("Confusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(report)
    
    return acc

accuracy = evaluate_model(model, X_test, y_test)

Accuracy: 0.9825 (98.25%)

Confusion Matrix:
[[41  1]
 [ 1 71]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [9]:
from sklearn.pipeline import Pipeline

X = df.drop(columns=['target'])
y = df['target']


X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

pipeline.fit(X_train_raw, y_train)
print(f"Pipeline Test Accuracy: {pipeline.score(X_test_raw, y_test):.4f}")

Pipeline Test Accuracy: 0.9825
